In [ ]:
"""
============================================================================
VOLATILITY FORECASTING FOR VOLATILITY-MANAGED PORTFOLIOS (VMP)
============================================================================
Models implemented:
  (a) Previous month realized volatility (RV 1-month)
  (b) Last 6-month realized volatility (RV 6-month)
  (c) GJR-GARCH(1,1) with Student-t innovations
  (d) EGARCH(1,1) with Student-t innovations
 
Evaluation:
  - RMSE, MAE, MAPE
  - Pearson correlation (forecast vs realized)
  - Mincer-Zarnowitz regression (unbiasedness test)
  - Diebold-Mariano test (pairwise comparison)
 
Input: CSV file with col1=dates, col2=prices (SPXT total return index)
============================================================================
"""


import pandas as pd
import numpy as np
from arch import arch_model
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import os

In [3]:
warnings.filterwarnings("ignore")
 
ANNUALIZATION_FACTOR = 252
INITIAL_ESTIMATION_WINDOW_DAYS = 1000  # ~4 years of trading days
LOOKBACK_SIX_MONTHS_DAYS = 126         # ~6 months of trading days

In [4]:
def load_price_data_and_compute_log_returns(csv_filepath, asset_to_forecast="SPXT"):
    """
    Charge toutes les colonnes, mais prépare 'asset_to_forecast' 
    pour l'analyse de volatilité.
    """
    # On lit tout le fichier pour avoir accès à tous les actifs
    df_all = pd.read_csv(csv_filepath)
    
    # On vérifie que l'actif demandé existe bien dans le fichier
    if asset_to_forecast not in df_all.columns:
        available = df_all.columns.tolist()
        raise ValueError(f"L'actif '{asset_to_forecast}' n'est pas dans le CSV. Colonnes dispos : {available}")

    # On ne garde que la date et l'actif choisi pour le reste du script
    df = df_all[["date", asset_to_forecast]].copy()
    df.columns = ["date", "price"]
    
    # Conversion et préparation (identique au script original)
    df["date"] = pd.to_datetime(df["date"], dayfirst=True)
    df = df.sort_values("date")
    df["year_month"] = df["date"].dt.to_period("M")
    
    # Calcul du rendement log pour l'actif choisi
    df["daily_log_return"] = np.log(df["price"] / df["price"].shift(1))
    
    return df.dropna(subset=["daily_log_return"])

In [5]:
# ====================================================================
# SECTION 2 — VOLATILITY FORECASTING (EXPANDING WINDOW)
# ====================================================================
 
def run_all_volatility_forecasts(daily_data):
    """
    For each month t (starting after the initial estimation window),
    produce one-step-ahead monthly volatility forecasts using four models.
 
    All forecasts are annualized: sigma_annual = sigma_daily * sqrt(252).
    GARCH models are re-estimated every month on the expanding sample.
    """
    all_year_months = daily_data["year_month"].unique()
 
    # --- Determine first forecastable month ---
    cumulative_trading_days_per_month = (
        daily_data.groupby("year_month").size().cumsum()
    )
 
    first_forecast_month_position = None
    for position, (month, cumulative_count) in enumerate(
        cumulative_trading_days_per_month.items()
    ):
        if cumulative_count >= INITIAL_ESTIMATION_WINDOW_DAYS:
            first_forecast_month_position = position + 1  # forecast the NEXT month
            break
 
    if (
        first_forecast_month_position is None
        or first_forecast_month_position >= len(all_year_months)
    ):
        raise ValueError(
            f"Not enough data. Need >= {INITIAL_ESTIMATION_WINDOW_DAYS} trading days "
            f"before the first forecast month."
        )
 
    months_to_forecast = all_year_months[first_forecast_month_position:]
    total_forecast_months = len(months_to_forecast)
 
    print("=" * 65)
    print("VOLATILITY FORECASTING — EXPANDING WINDOW")
    print("=" * 65)
    print(f"  Initial window    : {INITIAL_ESTIMATION_WINDOW_DAYS} trading days")
    print(f"  Forecast period   : {months_to_forecast[0]}  to  {months_to_forecast[-1]}")
    print(f"  Months to forecast: {total_forecast_months}")
    print()
 
    monthly_forecast_records = []
 
    for iteration_number, target_month in enumerate(months_to_forecast):
 
        # ----------------------------------------------------------
        # Gather available data (everything BEFORE target_month)
        # ----------------------------------------------------------
        available_data_before_target = daily_data[
            daily_data["year_month"] < target_month
        ]
        available_daily_log_returns = available_data_before_target[
            "daily_log_return"
        ].values
 
        # ----------------------------------------------------------
        # Realized vol of the target month (the "truth" we forecast)
        # ----------------------------------------------------------
        target_month_rows = daily_data[daily_data["year_month"] == target_month]
        if len(target_month_rows) < 10:
            continue  # skip incomplete months (e.g. last month of dataset)
 
        realized_volatility_annualized = (
            target_month_rows["daily_log_return"].std(ddof=1)
            * np.sqrt(ANNUALIZATION_FACTOR)
        )
 
        # ----------------------------------------------------------
        # MODEL (a) — Previous month realized volatility
        # ----------------------------------------------------------
        target_month_index = np.where(all_year_months == target_month)[0][0]
        previous_month = all_year_months[target_month_index - 1]
        previous_month_rows = daily_data[daily_data["year_month"] == previous_month]
 
        forecast_previous_month_realized_volatility = (
            previous_month_rows["daily_log_return"].std(ddof=1)
            * np.sqrt(ANNUALIZATION_FACTOR)
        )
 
        # ----------------------------------------------------------
        # MODEL (b) — Last 6-month realized volatility
        # ----------------------------------------------------------
        last_six_months_returns = available_daily_log_returns[
            -LOOKBACK_SIX_MONTHS_DAYS:
        ]
        forecast_six_month_realized_volatility = (
            np.std(last_six_months_returns, ddof=1) * np.sqrt(ANNUALIZATION_FACTOR)
        )
 
        # ----------------------------------------------------------
        # MODEL (c) — GJR-GARCH(1,1)
        # ----------------------------------------------------------
        forecast_gjr_garch_volatility = np.nan
        try:
            # Scale to percentage for numerical stability
            percentage_scaled_returns = available_daily_log_returns * 100
 
            gjr_garch_model_specification = arch_model(
                percentage_scaled_returns,
                vol="GARCH",
                p=1,
                o=1,   # o=1 adds the asymmetric (leverage) term → GJR
                q=1,
                dist="t",
                mean="Zero",
            )
            gjr_garch_fitted_result = gjr_garch_model_specification.fit(
                disp="off", show_warning=False
            )
 
            gjr_garch_one_step_forecast = gjr_garch_fitted_result.forecast(horizon=1)
            one_step_daily_variance_gjr = (
                gjr_garch_one_step_forecast.variance.iloc[-1, 0]
            )
 
            # Convert: sqrt(variance) * sqrt(252) / 100 (undo percentage scaling)
            forecast_gjr_garch_volatility = (
                np.sqrt(one_step_daily_variance_gjr)
                * np.sqrt(ANNUALIZATION_FACTOR)
                / 100
            )
 
        except Exception as gjr_estimation_error:
            pass  # silently skip; NaN recorded
 
        # ----------------------------------------------------------
        # MODEL (d) — EGARCH(1,1)
        # ----------------------------------------------------------
        forecast_egarch_volatility = np.nan
        try:
            percentage_scaled_returns = available_daily_log_returns * 100
 
            egarch_model_specification = arch_model(
                percentage_scaled_returns,
                vol="EGARCH",
                p=1,
                o=1,
                q=1,
                dist="t",
                mean="Zero",
            )
            egarch_fitted_result = egarch_model_specification.fit(
                disp="off", show_warning=False
            )
 
            egarch_one_step_forecast = egarch_fitted_result.forecast(horizon=1)
            one_step_daily_variance_egarch = (
                egarch_one_step_forecast.variance.iloc[-1, 0]
            )
 
            forecast_egarch_volatility = (
                np.sqrt(one_step_daily_variance_egarch)
                * np.sqrt(ANNUALIZATION_FACTOR)
                / 100
            )
 
        except Exception as egarch_estimation_error:
            pass
 
        # ----------------------------------------------------------
        # Store the record
        # ----------------------------------------------------------
        monthly_forecast_records.append(
            {
                "target_month": target_month,
                "month_end_date": target_month_rows["date"].max(),
                "trading_days_in_month": len(target_month_rows),
                "realized_volatility": realized_volatility_annualized,
                "forecast_previous_month_rv": forecast_previous_month_realized_volatility,
                "forecast_six_month_rv": forecast_six_month_realized_volatility,
                "forecast_gjr_garch": forecast_gjr_garch_volatility,
                "forecast_egarch": forecast_egarch_volatility,
                "estimation_sample_size_days": len(available_daily_log_returns),
            }
        )
 
        # Progress
        completed = iteration_number + 1
        if completed % 24 == 0 or completed == total_forecast_months:
            print(f"  [{completed:>4d} / {total_forecast_months}]  current month: {target_month}")
 
    forecast_results_dataframe = pd.DataFrame(monthly_forecast_records)
 
    # Count GARCH failures
    gjr_failures = forecast_results_dataframe["forecast_gjr_garch"].isna().sum()
    egarch_failures = forecast_results_dataframe["forecast_egarch"].isna().sum()
    print()
    print(f"  Forecasts generated : {len(forecast_results_dataframe)}")
    print(f"  GJR-GARCH failures  : {gjr_failures}")
    print(f"  EGARCH failures     : {egarch_failures}")
    print()
 
    return forecast_results_dataframe


In [6]:
 
# ====================================================================
# SECTION 3 — EVALUATION METRICS
# ====================================================================
 
MODEL_COLUMN_MAPPING = {
    "(a) RV 1-month":  "forecast_previous_month_rv",
    "(b) RV 6-month":  "forecast_six_month_rv",
    "(c) GJR-GARCH":   "forecast_gjr_garch",
    "(d) EGARCH":      "forecast_egarch",
}
 
 
def compute_single_model_evaluation_metrics(
    realized_values, forecast_values, model_name
):
    """
    Compute RMSE, MAE, MAPE, correlation, and Mincer-Zarnowitz
    regression for one model.
    """
    valid_mask = ~(np.isnan(realized_values) | np.isnan(forecast_values))
    realized_clean = realized_values[valid_mask]
    forecast_clean = forecast_values[valid_mask]
 
    if len(realized_clean) < 20:
        print(f"  WARNING: only {len(realized_clean)} valid obs for {model_name}")
        return None
 
    forecast_errors = realized_clean - forecast_clean
 
    # --- Error metrics ---
    root_mean_squared_error = np.sqrt(np.mean(forecast_errors**2))
    mean_absolute_error = np.mean(np.abs(forecast_errors))
    mean_absolute_percentage_error = (
        np.mean(np.abs(forecast_errors / realized_clean)) * 100
    )
 
    # --- Correlation ---
    pearson_correlation, pearson_pvalue = stats.pearsonr(
        realized_clean, forecast_clean
    )
 
    # --- Mincer-Zarnowitz regression ---
    #     RV_t  =  alpha  +  beta * forecast_t  +  epsilon_t
    #     H0 (unbiased):  alpha = 0  AND  beta = 1
    regression_dataframe = pd.DataFrame(
        {"realized": realized_clean, "forecast": forecast_clean}
    )
    mincer_zarnowitz_model = smf.ols(
        "realized ~ forecast", data=regression_dataframe
    ).fit()
 
    mz_intercept = mincer_zarnowitz_model.params["Intercept"]
    mz_slope = mincer_zarnowitz_model.params["forecast"]
    mz_r_squared = mincer_zarnowitz_model.rsquared
 
    mz_intercept_pvalue = mincer_zarnowitz_model.pvalues["Intercept"]
    mz_slope_pvalue = mincer_zarnowitz_model.pvalues["forecast"]
 
    # Joint F-test:  H0: alpha=0, beta=1
    try:
        joint_f_test_result = mincer_zarnowitz_model.f_test(
            "(Intercept = 0), (forecast = 1)"
        )
        mz_joint_f_statistic = float(joint_f_test_result.fvalue)
        mz_joint_f_pvalue = float(joint_f_test_result.pvalue)
    except Exception:
        mz_joint_f_statistic = np.nan
        mz_joint_f_pvalue = np.nan
 
    return {
        "model_name": model_name,
        "number_of_observations": len(realized_clean),
        "root_mean_squared_error": root_mean_squared_error,
        "mean_absolute_error": mean_absolute_error,
        "mean_absolute_percentage_error": mean_absolute_percentage_error,
        "pearson_correlation": pearson_correlation,
        "pearson_correlation_pvalue": pearson_pvalue,
        "mz_intercept_alpha": mz_intercept,
        "mz_slope_beta": mz_slope,
        "mz_r_squared": mz_r_squared,
        "mz_intercept_pvalue": mz_intercept_pvalue,
        "mz_slope_pvalue": mz_slope_pvalue,
        "mz_joint_f_statistic": mz_joint_f_statistic,
        "mz_joint_f_pvalue": mz_joint_f_pvalue,
    }
 
 
def compute_all_models_evaluation(forecast_results_dataframe):
    """
    Compute evaluation metrics for all four models and display results.
    """
    realized_values = forecast_results_dataframe["realized_volatility"].values
    all_metrics = []
 
    for model_label, column_name in MODEL_COLUMN_MAPPING.items():
        forecast_values = forecast_results_dataframe[column_name].values
        metrics = compute_single_model_evaluation_metrics(
            realized_values, forecast_values, model_label
        )
        if metrics is not None:
            all_metrics.append(metrics)
 
    metrics_dataframe = pd.DataFrame(all_metrics).set_index("model_name")
 
    # --- Print results ---
    print("=" * 85)
    print("FORECAST EVALUATION — ERROR METRICS")
    print("=" * 85)
    error_columns = [
        "number_of_observations",
        "root_mean_squared_error",
        "mean_absolute_error",
        "mean_absolute_percentage_error",
        "pearson_correlation",
    ]
    print(
        metrics_dataframe[error_columns].to_string(
            float_format=lambda x: f"{x:.4f}"
        )
    )
    print()
 
    print("=" * 85)
    print("FORECAST EVALUATION — MINCER-ZARNOWITZ REGRESSION")
    print("  RV_realized = alpha + beta * forecast + epsilon")
    print("  H0 (unbiased): alpha = 0  AND  beta = 1")
    print("=" * 85)
    mz_columns = [
        "mz_intercept_alpha",
        "mz_slope_beta",
        "mz_r_squared",
        "mz_intercept_pvalue",
        "mz_slope_pvalue",
        "mz_joint_f_statistic",
        "mz_joint_f_pvalue",
    ]
    print(
        metrics_dataframe[mz_columns].to_string(
            float_format=lambda x: f"{x:.4f}"
        )
    )
    print()
    print("  Interpretation:")
    print("    - If mz_joint_f_pvalue < 0.05 → forecast is statistically BIASED")
    print("    - mz_slope_beta > 1 → model UNDER-estimates vol on average")
    print("    - mz_slope_beta < 1 → model OVER-estimates vol on average")
    print("    - Higher mz_r_squared → better explanatory power")
    print()
 
    return metrics_dataframe

In [7]:
# ====================================================================
# SECTION 4 — DIEBOLD-MARIANO TEST
# ====================================================================
 
def compute_diebold_mariano_test(
    realized_values,
    forecast_values_model_1,
    forecast_values_model_2,
    model_name_1,
    model_name_2,
    loss_function="squared",
):
    """
    Diebold-Mariano test for equal predictive ability.
    H0: the two models have the same forecast accuracy.
 
    Uses squared-error loss by default.
    A negative DM statistic means model 1 is better (smaller loss).
    """
    valid_mask = ~(
        np.isnan(realized_values)
        | np.isnan(forecast_values_model_1)
        | np.isnan(forecast_values_model_2)
    )
    realized_clean = realized_values[valid_mask]
    forecast_1_clean = forecast_values_model_1[valid_mask]
    forecast_2_clean = forecast_values_model_2[valid_mask]
 
    if loss_function == "squared":
        loss_model_1 = (realized_clean - forecast_1_clean) ** 2
        loss_model_2 = (realized_clean - forecast_2_clean) ** 2
    elif loss_function == "absolute":
        loss_model_1 = np.abs(realized_clean - forecast_1_clean)
        loss_model_2 = np.abs(realized_clean - forecast_2_clean)
    else:
        raise ValueError("loss_function must be 'squared' or 'absolute'")
 
    loss_differential = loss_model_1 - loss_model_2
    number_of_observations = len(loss_differential)
    mean_loss_differential = np.mean(loss_differential)
 
    # Variance using Newey-West (lag=1 for monthly data)
    autocovariance_lag_0 = np.var(loss_differential, ddof=1)
    autocovariance_lag_1 = np.cov(
        loss_differential[1:], loss_differential[:-1]
    )[0, 1]
 
    # Newey-West long-run variance estimate (1 lag)
    long_run_variance = autocovariance_lag_0 + 2 * autocovariance_lag_1
 
    if long_run_variance <= 0:
        long_run_variance = autocovariance_lag_0  # fallback
 
    diebold_mariano_statistic = mean_loss_differential / np.sqrt(
        long_run_variance / number_of_observations
    )
    diebold_mariano_pvalue = 2 * (
        1 - stats.norm.cdf(np.abs(diebold_mariano_statistic))
    )
 
    return {
        "model_1": model_name_1,
        "model_2": model_name_2,
        "dm_statistic": diebold_mariano_statistic,
        "dm_pvalue": diebold_mariano_pvalue,
        "mean_loss_differential": mean_loss_differential,
        "preferred_model": (
            model_name_1
            if mean_loss_differential < 0
            else model_name_2
        ),
    }
 
 
def run_all_pairwise_diebold_mariano_tests(forecast_results_dataframe):
    """
    Run Diebold-Mariano tests for all pairs of models.
    """
    realized_values = forecast_results_dataframe["realized_volatility"].values
    model_names = list(MODEL_COLUMN_MAPPING.keys())
    model_columns = list(MODEL_COLUMN_MAPPING.values())
 
    pairwise_results = []
    for i in range(len(model_names)):
        for j in range(i + 1, len(model_names)):
            forecast_values_i = forecast_results_dataframe[model_columns[i]].values
            forecast_values_j = forecast_results_dataframe[model_columns[j]].values
 
            dm_result = compute_diebold_mariano_test(
                realized_values,
                forecast_values_i,
                forecast_values_j,
                model_names[i],
                model_names[j],
            )
            pairwise_results.append(dm_result)
 
    dm_results_dataframe = pd.DataFrame(pairwise_results)
 
    print("=" * 85)
    print("DIEBOLD-MARIANO TEST — PAIRWISE FORECAST COMPARISON")
    print("  H0: both models have equal predictive ability  (squared-error loss)")
    print("  DM < 0 → model_1 is better  |  DM > 0 → model_2 is better")
    print("=" * 85)
    for _, row in dm_results_dataframe.iterrows():
        significance = ""
        if row["dm_pvalue"] < 0.01:
            significance = "***"
        elif row["dm_pvalue"] < 0.05:
            significance = "**"
        elif row["dm_pvalue"] < 0.10:
            significance = "*"
 
        print(
            f"  {row['model_1']:>18s}  vs  {row['model_2']:<18s}  |"
            f"  DM = {row['dm_statistic']:+7.3f}  "
            f"  p = {row['dm_pvalue']:.4f} {significance:>3s}  "
            f"  →  {row['preferred_model']}"
        )
    print()
    print("  Significance: *** p<0.01   ** p<0.05   * p<0.10")
    print()
 
    return dm_results_dataframe

In [8]:
# ====================================================================
# SECTION 5 — VISUALIZATIONS
# ====================================================================
 
def create_all_forecast_plots(forecast_results_dataframe, output_directory):
    """
    Generate and save four types of plots:
      1. Time series: forecasted vs realized volatility
      2. Scatter plots: forecast vs realized (with MZ regression line)
      3. Bar chart: RMSE comparison across models
      4. Forecast error over time
    """
    os.makedirs(output_directory, exist_ok=True)
 
    dates_for_plotting = forecast_results_dataframe["month_end_date"].values
    realized_volatility = forecast_results_dataframe["realized_volatility"].values
 
    model_colors = {
        "(a) RV 1-month": "#1f77b4",
        "(b) RV 6-month": "#ff7f0e",
        "(c) GJR-GARCH":  "#2ca02c",
        "(d) EGARCH":     "#d62728",
    }
 
    # ------------------------------------------------------------------
    # PLOT 1: Time series — all forecasts vs realized
    # ------------------------------------------------------------------
    figure_timeseries, axes_timeseries = plt.subplots(
        2, 2, figsize=(16, 10), sharex=True, sharey=True
    )
    axes_flat = axes_timeseries.flatten()
 
    for plot_index, (model_label, column_name) in enumerate(
        MODEL_COLUMN_MAPPING.items()
    ):
        current_axis = axes_flat[plot_index]
        forecast_values = forecast_results_dataframe[column_name].values
 
        current_axis.plot(
            dates_for_plotting,
            realized_volatility,
            color="black",
            alpha=0.5,
            linewidth=0.8,
            label="Realized volatility",
        )
        current_axis.plot(
            dates_for_plotting,
            forecast_values,
            color=model_colors[model_label],
            linewidth=1.0,
            alpha=0.8,
            label=model_label,
        )
        current_axis.set_title(model_label, fontsize=11, fontweight="bold")
        current_axis.legend(loc="upper right", fontsize=8)
        current_axis.set_ylabel("Annualized volatility")
        current_axis.grid(True, alpha=0.3)
 
    figure_timeseries.suptitle(
        "Volatility Forecasts vs Realized Volatility (annualized)",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    timeseries_filepath = os.path.join(
        output_directory, "plot_forecast_vs_realized_timeseries.png"
    )
    figure_timeseries.savefig(timeseries_filepath, dpi=150, bbox_inches="tight")
    plt.close(figure_timeseries)
    print(f"  Saved: {timeseries_filepath}")
 
    # ------------------------------------------------------------------
    # PLOT 2: Scatter plots with MZ regression line
    # ------------------------------------------------------------------
    figure_scatter, axes_scatter = plt.subplots(2, 2, figsize=(14, 12))
    axes_flat_scatter = axes_scatter.flatten()
 
    for plot_index, (model_label, column_name) in enumerate(
        MODEL_COLUMN_MAPPING.items()
    ):
        current_axis = axes_flat_scatter[plot_index]
        forecast_values = forecast_results_dataframe[column_name].values
 
        valid_mask = ~(np.isnan(realized_volatility) | np.isnan(forecast_values))
        realized_clean = realized_volatility[valid_mask]
        forecast_clean = forecast_values[valid_mask]
 
        current_axis.scatter(
            forecast_clean,
            realized_clean,
            alpha=0.4,
            s=15,
            color=model_colors[model_label],
        )
 
        # 45-degree line (perfect forecast)
        axis_min = min(forecast_clean.min(), realized_clean.min()) * 0.9
        axis_max = max(forecast_clean.max(), realized_clean.max()) * 1.1
        current_axis.plot(
            [axis_min, axis_max],
            [axis_min, axis_max],
            "k--",
            alpha=0.4,
            label="45° line (perfect)",
        )
 
        # MZ regression line
        regression_data = pd.DataFrame(
            {"realized": realized_clean, "forecast": forecast_clean}
        )
        mz_fit = smf.ols("realized ~ forecast", data=regression_data).fit()
        x_line = np.linspace(axis_min, axis_max, 100)
        y_line = mz_fit.params["Intercept"] + mz_fit.params["forecast"] * x_line
        current_axis.plot(
            x_line,
            y_line,
            color="red",
            linewidth=1.5,
            alpha=0.7,
            label=f"MZ: α={mz_fit.params['Intercept']:.3f}, β={mz_fit.params['forecast']:.3f}, R²={mz_fit.rsquared:.3f}",
        )
 
        current_axis.set_xlabel("Forecast volatility")
        current_axis.set_ylabel("Realized volatility")
        current_axis.set_title(model_label, fontsize=11, fontweight="bold")
        current_axis.legend(fontsize=7, loc="upper left")
        current_axis.set_xlim(axis_min, axis_max)
        current_axis.set_ylim(axis_min, axis_max)
        current_axis.set_aspect("equal")
        current_axis.grid(True, alpha=0.3)
 
    figure_scatter.suptitle(
        "Mincer-Zarnowitz Scatter: Forecast vs Realized Volatility",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    scatter_filepath = os.path.join(
        output_directory, "plot_mincer_zarnowitz_scatter.png"
    )
    figure_scatter.savefig(scatter_filepath, dpi=150, bbox_inches="tight")
    plt.close(figure_scatter)
    print(f"  Saved: {scatter_filepath}")
 
    # ------------------------------------------------------------------
    # PLOT 3: Bar chart — RMSE comparison
    # ------------------------------------------------------------------
    rmse_values = {}
    for model_label, column_name in MODEL_COLUMN_MAPPING.items():
        forecast_values = forecast_results_dataframe[column_name].values
        valid_mask = ~(np.isnan(realized_volatility) | np.isnan(forecast_values))
        errors = realized_volatility[valid_mask] - forecast_values[valid_mask]
        rmse_values[model_label] = np.sqrt(np.mean(errors**2))
 
    figure_bar, axis_bar = plt.subplots(figsize=(10, 6))
    bar_names = list(rmse_values.keys())
    bar_vals = list(rmse_values.values())
    bar_colors = [model_colors[name] for name in bar_names]
 
    bars = axis_bar.bar(bar_names, bar_vals, color=bar_colors, alpha=0.8, edgecolor="black")
    for bar, value in zip(bars, bar_vals):
        axis_bar.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{value:.4f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )
 
    axis_bar.set_ylabel("RMSE (annualized volatility)")
    axis_bar.set_title(
        "Forecast Accuracy — Root Mean Squared Error",
        fontsize=13,
        fontweight="bold",
    )
    axis_bar.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    bar_filepath = os.path.join(output_directory, "plot_rmse_comparison.png")
    figure_bar.savefig(bar_filepath, dpi=150, bbox_inches="tight")
    plt.close(figure_bar)
    print(f"  Saved: {bar_filepath}")
 
    # ------------------------------------------------------------------
    # PLOT 4: Forecast errors over time
    # ------------------------------------------------------------------
    figure_errors, axes_errors = plt.subplots(
        2, 2, figsize=(16, 10), sharex=True, sharey=True
    )
    axes_flat_errors = axes_errors.flatten()
 
    for plot_index, (model_label, column_name) in enumerate(
        MODEL_COLUMN_MAPPING.items()
    ):
        current_axis = axes_flat_errors[plot_index]
        forecast_values = forecast_results_dataframe[column_name].values
        forecast_errors = realized_volatility - forecast_values
 
        current_axis.bar(
            dates_for_plotting,
            forecast_errors,
            color=model_colors[model_label],
            alpha=0.6,
            width=25,
        )
        current_axis.axhline(y=0, color="black", linewidth=0.8)
        current_axis.set_title(model_label, fontsize=11, fontweight="bold")
        current_axis.set_ylabel("Forecast error (realized − forecast)")
        current_axis.grid(True, alpha=0.3)
 
    figure_errors.suptitle(
        "Forecast Errors Over Time  (positive = model under-estimated)",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    errors_filepath = os.path.join(
        output_directory, "plot_forecast_errors_over_time.png"
    )
    figure_errors.savefig(errors_filepath, dpi=150, bbox_inches="tight")
    plt.close(figure_errors)
    print(f"  Saved: {errors_filepath}")

In [11]:
# ====================================================================
# SECTION 6 — MAIN EXECUTION
# ====================================================================
 
def main():
    # ------------------------------------------------------------------
    # CONFIGURATION — change this path to your CSV file
    # ------------------------------------------------------------------
    csv_filepath = "bloomberg_data_clean.csv"
 
    # ------------------------------------------------------------------
    # STEP 1: Load data and compute returns
    # ------------------------------------------------------------------
    daily_data = load_price_data_and_compute_log_returns(csv_filepath)
 
    # ------------------------------------------------------------------
    # STEP 2: Run volatility forecasts
    # ------------------------------------------------------------------
    forecast_results = run_all_volatility_forecasts(daily_data)
 
    # ------------------------------------------------------------------
    # STEP 3: Evaluate forecasts
    # ------------------------------------------------------------------
    evaluation_metrics = compute_all_models_evaluation(forecast_results)
 
    # ------------------------------------------------------------------
    # STEP 4: Diebold-Mariano tests
    # ------------------------------------------------------------------
    diebold_mariano_results = run_all_pairwise_diebold_mariano_tests(
        forecast_results
    )
 
    # ------------------------------------------------------------------
    # STEP 5: Generate plots
    # ------------------------------------------------------------------
    output_directory = ""


    print("=" * 65)
    print("GENERATING PLOTS")
    print("=" * 65)
    create_all_forecast_plots(forecast_results, output_directory)
 
    # ------------------------------------------------------------------
    # STEP 6: Save results to CSV
    # ------------------------------------------------------------------
    forecast_output_path = os.path.join(
        output_directory, "volatility_forecast_results.csv"
    )
    forecast_results_to_save = forecast_results.copy()
    forecast_results_to_save["target_month"] = forecast_results_to_save[
        "target_month"
    ].astype(str)
    forecast_results_to_save.to_csv(forecast_output_path, index=False)
    print(f"\n  Forecast data saved to: {forecast_output_path}")
 
    metrics_output_path = os.path.join(
        output_directory, "evaluation_metrics.csv"
    )
    evaluation_metrics.to_csv(metrics_output_path)
    print(f"  Metrics saved to: {metrics_output_path}")
 
    dm_output_path = os.path.join(
        output_directory, "diebold_mariano_tests.csv"
    )
    diebold_mariano_results.to_csv(dm_output_path, index=False)
    print(f"  DM tests saved to: {dm_output_path}")
 
    print()
    print("=" * 65)
    print("ALL DONE")
    print("=" * 65)

In [12]:
main()

VOLATILITY FORECASTING — EXPANDING WINDOW
  Initial window    : 1000 trading days
  Forecast period   : 1999-01  to  2025-12
  Months to forecast: 324

  [  24 / 324]  current month: 2000-12
  [  48 / 324]  current month: 2002-12
  [  72 / 324]  current month: 2004-12
  [  96 / 324]  current month: 2006-12
  [ 120 / 324]  current month: 2008-12
  [ 144 / 324]  current month: 2010-12
  [ 168 / 324]  current month: 2012-12
  [ 192 / 324]  current month: 2014-12
  [ 216 / 324]  current month: 2016-12
  [ 240 / 324]  current month: 2018-12
  [ 264 / 324]  current month: 2020-12
  [ 288 / 324]  current month: 2022-12
  [ 312 / 324]  current month: 2024-12
  [ 324 / 324]  current month: 2025-12

  Forecasts generated : 324
  GJR-GARCH failures  : 0
  EGARCH failures     : 0

FORECAST EVALUATION — ERROR METRICS
                number_of_observations  root_mean_squared_error  mean_absolute_error  mean_absolute_percentage_error  pearson_correlation
model_name                                    

FileNotFoundError: [WinError 3] The system cannot find the path specified: ''